# 01 · Compute normalisation stats (train split only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1ow1atency/floodfusion/blob/main/notebooks/01_compute_stats.ipynb)

Runs `scripts/compute_stats.py` on the full data on Drive. Writes `band_stats.json` (per-band clip limits, mean, std from **train only**, used by `Sen1Floods11(stats=...)`) and `label_stats.json` (descriptive label counts per split, not used for training). Needs `00_download_data.ipynb` to have run first. CPU runtime is enough.

Mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone the repo, or pull if it is already cloned. Uses the `GITHUB_TOKEN` Colab secret if set (see docs/colab_setup.md); otherwise tries a plain public clone.

In [ ]:
# Clone or update the repo in Colab. Shared cell: paste verbatim into every notebook
# (tests/test_notebooks.py checks the copies match notebooks/colab_clone_cell.py).
# Uses the Colab secret GITHUB_TOKEN if available (see docs/colab_setup.md). The token
# reaches git only through environment variables: never on the command line, never in
# .git/config (the remote URL stays token-free), and it is masked in any output.
import base64, os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/1ow1atency/floodfusion.git'
REPO_DIR = Path('/content/floodfusion')


def _github_token():
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception as e:  # secret missing, notebook access off, or not in Colab
        print(f'GITHUB_TOKEN secret not available ({type(e).__name__}); trying a plain public clone.\n'
              'To add it: key icon in the left sidebar -> "Add new secret" -> name GITHUB_TOKEN, value = a\n'
              'GitHub token with read access to this repo -> switch "Notebook access" on -> re-run this cell.')
        return None
    return token.strip() or None


def _git(*args, token=None):
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}  # fail instead of waiting for a password
    secrets = []
    if token:
        b64 = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0=f'Authorization: Basic {b64}')
        secrets = [token, b64]
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode != 0:
        hint = '' if token else '\n(No token used. If the repo is not publicly visible, add the GITHUB_TOKEN secret.)'
        raise RuntimeError(f'git {args[0] if args[0] != "-C" else args[2]} failed:\n{out}{hint}')
    return out


def _clone_or_pull():
    version = tuple(int(x) for x in _git('--version').split()[2].split('.')[:2])
    if version < (2, 31):  # GIT_CONFIG_COUNT needs git >= 2.31
        raise RuntimeError(f'git >= 2.31 needed, found {version}')
    token = _github_token()
    if (REPO_DIR / '.git').exists():
        _git('-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL)  # keep the remote token-free
        print(_git('-C', str(REPO_DIR), 'pull', '--ff-only', token=token))
    else:
        print(_git('clone', REPO_URL, str(REPO_DIR), token=token))
    print('at', _git('-C', str(REPO_DIR), 'log', '--oneline', '-1'))


_clone_or_pull()

Install the package (editable).

In [ ]:
!pip install -q -e {REPO_DIR}
assert _exit_code == 0, 'pip install failed, see output above'

Compute the stats. No `--allow-missing`: on the full data any missing file must be an error.

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/floodfusion/data/sen1floods11'
OUT = '/content/drive/MyDrive/floodfusion/outputs/band_stats.json'
SEED = 0

!cd {REPO_DIR} && python scripts/compute_stats.py --root "{DATA_ROOT}" --out "{OUT}" --seed {SEED}
assert _exit_code == 0, 'compute_stats.py failed, see output above'

Print both JSON files.

In [ ]:
for path in [Path(OUT), Path(OUT).with_name('label_stats.json')]:
    print(f'===== {path} =====')
    print(path.read_text())